In [1]:
!pip install -q langchain-community faiss-cpu langchain-huggingface sentence-transformers langchain-classic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

/tmp/ipykernel_459/1289553017.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=(
            "RAG, or Retrieval-Augmented Generation, gives a language model documents "
            "retrieved at question time, so it can answer from your own data instead of "
            "only from its training."
        ),
        metadata={"topic": "rag", "level": "beginner"},
    ),
    Document(
        page_content=(
            "An embedding turns text into a list of numbers that captures its meaning. "
            "Texts with similar meaning get vectors that are close together."
        ),
        metadata={"topic": "embeddings", "level": "beginner"},
    ),
    Document(
        page_content=(
            "A vector store such as FAISS, Chroma or Pinecone stores embeddings with their "
            "text and finds the vectors closest to a query."
        ),
        metadata={"topic": "vector store", "level": "beginner"},
    ),
    Document(
        page_content=(
            "Chunking splits long documents into smaller pieces before embedding. "
            "Chunk size and chunk overlap control how much text each piece holds and how "
            "much is repeated between neighbours."
        ),
        metadata={"topic": "chunking", "level": "beginner"},
    ),
    Document(
        page_content=(
            "A retriever takes a question and returns the most relevant documents. "
            "Any vector store can become one with the as_retriever method."
        ),
        metadata={"topic": "retriever", "level": "beginner"},
    ),
    Document(
        page_content=(
            "Maximal Marginal Relevance, or MMR, returns results that are relevant and "
            "different from each other, which reduces repeated answers."
        ),
        metadata={"topic": "retriever", "level": "intermediate"},
    ),
    Document(
        page_content=(
            "The multi-query retriever asks a language model to rewrite the question in "
            "several ways, searches for each version, and merges the unique results."
        ),
        metadata={"topic": "retriever", "level": "intermediate"},
    ),
    Document(
        page_content=(
            "Hybrid search combines keyword matching, such as BM25, with semantic vector "
            "search, so exact terms and similar meanings are both found."
        ),
        metadata={"topic": "retriever", "level": "intermediate"},
    ),
    Document(
        page_content=(
            "Hallucination is when a language model states something false with confidence. "
            "Grounding answers in retrieved documents is one way to reduce it."
        ),
        metadata={"topic": "rag", "level": "beginner"},
    ),
    Document(
        page_content=(
            "The context window is the maximum amount of text a model can read at once, "
            "so a RAG system sends only the best retrieved chunks to the prompt."
        ),
        metadata={"topic": "llm", "level": "intermediate"},
    ),
]

print(len(docs))

10


In [4]:
# Initialize embeddings and the FAISS vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(documents=docs, embedding=embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Retriever ---> Similarity
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [6]:
# Free local LLM (no API key). The first run downloads about 3 GB, so it takes a few minutes.
# Tip: Runtime > Change runtime type > T4 GPU makes it much faster.
import re
import torch
from transformers import pipeline
from langchain_core.runnables import RunnableLambda

device = 0 if torch.cuda.is_available() else -1
print("Running on:", "GPU" if device == 0 else "CPU")

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",   # for a faster, weaker model use "Qwen/Qwen2.5-0.5B-Instruct"
    device=device,
)

def local_llm(prompt_value):
    # accepts a LangChain prompt value or a plain string
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    out = generator(
        [{"role": "user", "content": text}],
        max_new_tokens=120,
        do_sample=False,
    )
    reply = out[0]["generated_text"][-1]["content"]
    # clean up: remove numbering/bullets, drop very short lines, keep at most 3 lines
    cleaned = [re.sub(r"^\s*(?:\d+[.)]|[-*\u2022])\s*", "", l).strip() for l in reply.splitlines()]
    cleaned = [l for l in cleaned if len(l) > 10]
    return "\n".join(cleaned[:3])

llm = RunnableLambda(local_llm)

# quick test of the LLM
print(llm.invoke("Rewrite this question in 3 different ways, one per line: How do I stop a model from inventing facts?"))

Running on: CPU


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

What strategies can be employed to prevent a model from generating false information?
How can we ensure that the output of a model is accurate and reliable?
In what methods can we guarantee that a model does not produce incorrect or fabricated data?


In [7]:
# MultiQuery Retriever
from langchain_core.prompts import PromptTemplate

query_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        "Rewrite the following question in 3 different ways to search a document database.\n"
        "Write exactly 3 lines, one question per line, with no numbering and no extra text.\n\n"
        "Question: {question}"
    ),
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
    llm=llm,
    prompt=query_prompt,
    include_original=True,   # always search the original question too, a safety net for small models
)

In [8]:
# Compare similarity retriever vs multi-query retriever
import logging
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)  # prints the rewritten questions

def show(retriever, query):
    print("Query:", query)
    for i, d in enumerate(retriever.invoke(query), 1):
        print(f"  {i}. [{d.metadata.get('topic')}] {d.page_content[:70]}...")

query = "How do I stop a model from inventing facts?"

print("SIMILARITY")
show(similarity_retriever, query)

print("\nMULTI-QUERY")
show(multi_query_retriever, query)

SIMILARITY
Query: How do I stop a model from inventing facts?
  1. [rag] Hallucination is when a language model states something false with con...
  2. [embeddings] An embedding turns text into a list of numbers that captures its meani...

MULTI-QUERY
Query: How do I stop a model from inventing facts?


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What strategies can be employed to prevent a machine learning model from generating false information?', 'How can one ensure that a predictive model does not produce inaccurate predictions or misinformation?', 'In what methods can we implement safeguards to avoid a neural network from producing erroneous data points?']


  1. [rag] Hallucination is when a language model states something false with con...
  2. [rag] RAG, or Retrieval-Augmented Generation, gives a language model documen...
  3. [vector store] A vector store such as FAISS, Chroma or Pinecone stores embeddings wit...
  4. [embeddings] An embedding turns text into a list of numbers that captures its meani...
